# Spark RDD Project: Log Analytics, Fraud Detection, and API Monitoring

## Problem Statement
In a large e-commerce company, multiple systems generate huge volumes of logs every day. These logs contain user activity, authentication events, payment transactions, and API performance details. The company wants to analyze these logs to detect suspicious activity, identify repeated login and payment failures, monitor slow APIs, and generate meaningful business and operational insights.

In this project, we will use Apache Spark RDD to:
- read multiple log files
- parse raw log records
- clean malformed data
- analyze user actions and errors
- detect fraud and suspicious patterns
- monitor API performance
- build final alert and summary reports

In [5]:
from pyspark import SparkContext

# Create SparkContext
#sc = SparkContext("local[*]", "RDD_Log_Analytics_Project")

# Read all files as RDDs
app_logs = sc.textFile("app_logs.txt")
auth_logs = sc.textFile("auth_logs.txt")
payment_logs = sc.textFile("payments_logs.txt")
api_logs = sc.textFile("api_gateway_logs.txt")

users_ref = sc.textFile("users_ref.csv")
ip_blacklist = sc.textFile("ip_blacklist.txt")

print("App Logs Sample :")
for row in app_logs.take(3):
    print(row)

print("\nRecords Count:")
print("App Logs Count:",app_logs.count())

print("\nPartitions:")
print("App logs Partitions:",app_logs.getNumPartitions())


App Logs Sample :
2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200
2026-03-05T10:00:12Z level=INFO user=user123 action=VIEW_PRODUCT ip=192.168.1.10 device=web endpoint=/product/432 ms=180 status=200
2026-03-05T10:00:20Z level=INFO user=user222 action=LOGIN_SUCCESS ip=192.168.1.12 device=android endpoint=/login ms=140 status=200

Records Count:
App Logs Count: 43

Partitions:
App logs Partitions: 2


In [13]:
def parse_kv_log(line):
    # create dictionary
    #2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200
    parts=line.split()
    print(parts)
    record={}
    record["ts"]=parts[0]
    for item in parts[1:]:
        if "=" in item:
            key,value=item.split("=",1)
            record[key]=value
    return record

app_parsed=app_logs.map(parse_kv_log)

In [14]:
app_parsed

PythonRDD[53] at RDD at PythonRDD.scala:53

In [ ]:
#line="2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200"
parts=line.split()
print(parts)
record={}
record["ts"]=parts[0]
for item in parts[1:]:
    if "=" in item:
        key,value=item.split("=",1)
        record[key]=value



['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']


In [10]:
record

{'ts': '2026-03-05T10:00:05Z',
 'level': 'INFO',
 'user': 'user123',
 'action': 'LOGIN_SUCCESS',
 'ip': '192.168.1.10',
 'device': 'web',
 'endpoint': '/login',
 'ms': '120',
 'status': '200'}

In [15]:
# for i in app_logs:
#     print(i)
print("parsed logs:")
for row in app_parsed.take(3):
    print(row)

parsed logs:
{'ts': '2026-03-05T10:00:05Z', 'level': 'INFO', 'user': 'user123', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.10', 'device': 'web', 'endpoint': '/login', 'ms': '120', 'status': '200'}
{'ts': '2026-03-05T10:00:12Z', 'level': 'INFO', 'user': 'user123', 'action': 'VIEW_PRODUCT', 'ip': '192.168.1.10', 'device': 'web', 'endpoint': '/product/432', 'ms': '180', 'status': '200'}
{'ts': '2026-03-05T10:00:20Z', 'level': 'INFO', 'user': 'user222', 'action': 'LOGIN_SUCCESS', 'ip': '192.168.1.12', 'device': 'android', 'endpoint': '/login', 'ms': '140', 'status': '200'}


['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']


In [16]:
for row in app_parsed:
    print(row)

TypeError: 'PipelinedRDD' object is not iterable

In [17]:
app_parsed.take(3)

['2026-03-05T10:00:05Z', 'level=INFO', 'user=user123', 'action=LOGIN_SUCCESS', 'ip=192.168.1.10', 'device=web', 'endpoint=/login', 'ms=120', 'status=200']
['2026-03-05T10:00:12Z', 'level=INFO', 'user=user123', 'action=VIEW_PRODUCT', 'ip=192.168.1.10', 'device=web', 'endpoint=/product/432', 'ms=180', 'status=200']
['2026-03-05T10:00:20Z', 'level=INFO', 'user=user222', 'action=LOGIN_SUCCESS', 'ip=192.168.1.12', 'device=android', 'endpoint=/login', 'ms=140', 'status=200']


[{'ts': '2026-03-05T10:00:05Z',
  'level': 'INFO',
  'user': 'user123',
  'action': 'LOGIN_SUCCESS',
  'ip': '192.168.1.10',
  'device': 'web',
  'endpoint': '/login',
  'ms': '120',
  'status': '200'},
 {'ts': '2026-03-05T10:00:12Z',
  'level': 'INFO',
  'user': 'user123',
  'action': 'VIEW_PRODUCT',
  'ip': '192.168.1.10',
  'device': 'web',
  'endpoint': '/product/432',
  'ms': '180',
  'status': '200'},
 {'ts': '2026-03-05T10:00:20Z',
  'level': 'INFO',
  'user': 'user222',
  'action': 'LOGIN_SUCCESS',
  'ip': '192.168.1.12',
  'device': 'android',
  'endpoint': '/login',
  'ms': '140',
  'status': '200'}]

In [18]:
app_logs.take(3)

['2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200',
 '2026-03-05T10:00:12Z level=INFO user=user123 action=VIEW_PRODUCT ip=192.168.1.10 device=web endpoint=/product/432 ms=180 status=200',
 '2026-03-05T10:00:20Z level=INFO user=user222 action=LOGIN_SUCCESS ip=192.168.1.12 device=android endpoint=/login ms=140 status=200']

In [20]:
for i in app_logs.collect():
    print(i)

2026-03-05T10:00:05Z level=INFO user=user123 action=LOGIN_SUCCESS ip=192.168.1.10 device=web endpoint=/login ms=120 status=200
2026-03-05T10:00:12Z level=INFO user=user123 action=VIEW_PRODUCT ip=192.168.1.10 device=web endpoint=/product/432 ms=180 status=200
2026-03-05T10:00:20Z level=INFO user=user222 action=LOGIN_SUCCESS ip=192.168.1.12 device=android endpoint=/login ms=140 status=200
2026-03-05T10:00:31Z level=INFO user=user222 action=SEARCH ip=192.168.1.12 device=android endpoint=/search ms=410 status=200
2026-03-05T10:01:02Z level=WARN user=user999 action=API_CALL ip=192.168.1.99 device=web endpoint=/search ms=3100 status=200
2026-03-05T10:01:13Z level=INFO user=user123 action=ADD_TO_CART ip=192.168.1.10 device=web endpoint=/cart/add ms=220 status=200
2026-03-05T10:01:40Z level=INFO user=user333 action=LOGIN_SUCCESS ip=192.168.1.45 device=ios endpoint=/login ms=110 status=200
2026-03-05T10:01:55Z level=INFO user=user333 action=VIEW_PRODUCT ip=192.168.1.45 device=ios endpoint=/prod